In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torch.optim as optim
from torch.utils.data import DataLoader
import time
import math
from tqdm import tqdm
from torchmetrics.functional import calibration_error

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")

✅ Using device: cuda


In [3]:
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)

testset = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False, num_workers=2, pin_memory=True)

In [4]:
# class EfficientNetB0_CIFAR100(nn.Module):
#     def __init__(self, num_classes=100):
#         super().__init__()
#         self.base = torchvision.models.efficientnet_b0(weights=None)
#         # Adjust first conv if input smaller
#         self.base.features[0][0] = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1, bias=False)
#         self.base.classifier = nn.Sequential(
#             nn.Linear(1280, num_classes)
#         )

#     def forward(self, x):
#         x = self.base.features(x)
#         x = F.adaptive_avg_pool2d(x, (1,1))
#         x = torch.flatten(x, 1)
#         x = self.base.classifier(x)
#         return x

# model = EfficientNetB0_CIFAR100().to(device)
model = torchvision.models.efficientnet_b0(weights=None, num_classes=100).to(device)
criterion = nn.CrossEntropyLoss()
#optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-4)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)


In [5]:
def topk_accuracy(output, target, topk=(1,)):
    maxk = max(topk)
    _, pred = output.topk(maxk, 1, True, True)
    correct = pred.eq(target.view(-1,1).expand_as(pred))
    res = []
    for k in topk:
        correct_k = correct[:, :k].sum().item()
        res.append(correct_k)
    return res

def mean_off_diagonal_covariance(features, eps=1e-8):
    X = features - features.mean(dim=0, keepdim=True)
    cov = (X.T @ X) / (X.shape[0]-1)
    var = cov.diag()
    denom = torch.sqrt(var.unsqueeze(0)*var.unsqueeze(1)).clamp_min(eps)
    corr = cov/denom
    corr.fill_diagonal_(0)
    return corr.abs().mean()

def linear_CKA(X, Y):
    X = X - X.mean(0, keepdim=True)
    Y = Y - Y.mean(0, keepdim=True)
    hsic = torch.norm(X.T @ Y, p='fro')**2
    var1 = torch.norm(X.T @ X, p='fro') * torch.norm(Y.T @ Y, p='fro')
    return (hsic / var1).item()

In [6]:
# =========================================================
# TRAINING
# =========================================================
num_epochs = 100
best_acc = 0
best_val_loss = float('inf')
convergence_epoch = None
patience = 5
no_improve_epochs = 0
train_losses, test_losses = [], []
start_time = time.time()

for epoch in range(num_epochs):
    model.train()
    running_loss, total, correct_top1, correct_top5 = 0.0, 0, 0, 0
    epoch_start = time.time()

    for inputs, labels in tqdm(trainloader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()
        top1, top5 = topk_accuracy(outputs, labels, topk=(1, 5))
        correct_top1 += top1
        correct_top5 += top5
        total += labels.size(0)

    train_loss = running_loss / len(trainloader)
    train_acc1 = 100 * correct_top1 / total
    train_acc5 = 100 * correct_top5 / total
    train_losses.append(train_loss)

    # =====================================================
    # VALIDATION
    # =====================================================
    model.eval()
    correct_top1, correct_top5, total = 0, 0, 0
    test_loss = 0.0
    probs, targets, features = [], [], []

    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            test_loss += loss.item()

            top1, top5 = topk_accuracy(outputs, labels, topk=(1, 5))
            correct_top1 += top1
            correct_top5 += top5
            total += labels.size(0)

            probs.append(F.softmax(outputs, dim=1).cpu())
            targets.append(labels.cpu())

            # Use features before classifier for redundancy metrics
            feats = model.features(inputs).view(inputs.size(0), -1).cpu()
            features.append(feats)

    test_loss /= len(testloader)
    test_losses.append(test_loss)
    test_acc1 = 100 * correct_top1 / total
    test_acc5 = 100 * correct_top5 / total
    train_test_gap = abs(train_acc1 - test_acc1)

    all_probs = torch.cat(probs)
    all_targets = torch.cat(targets)
    ece = calibration_error(all_probs, all_targets, n_bins=15, task="multiclass", num_classes=100).item()

    all_features = torch.cat(features, dim=0)
    moc = mean_off_diagonal_covariance(all_features)

    epoch_time = time.time() - epoch_start

    print(f"\n📊 Epoch {epoch+1}:")
    print(f"Train Loss: {train_loss:.4f} | Train Acc@1: {train_acc1:.2f}% | Train Acc@5: {train_acc5:.2f}%")
    print(f"Val Loss: {test_loss:.4f} | Val Acc@1: {test_acc1:.2f}% | Val Acc@5: {test_acc5:.2f}%")
    print(f"Train–Test Gap: {train_test_gap:.2f}% | ECE: {ece:.4f} | MOC: {moc:.4f}")
    print(f"⏱ Epoch Time: {epoch_time:.2f}s")

    # Early stopping
    if test_loss < best_val_loss - 1e-4:
        best_val_loss = test_loss
        no_improve_epochs = 0
    else:
        no_improve_epochs += 1

    if no_improve_epochs >= patience:
        print(f"\n⏹ Early stopping triggered after {epoch+1} epochs (no improvement in {patience} epochs).")
        break

    if test_acc1 > best_acc:
        best_acc = test_acc1
        convergence_epoch = epoch + 1
        torch.save(model.state_dict(), "best_efficientnetb0_cifar100_mps.pth")

total_time = time.time() - start_time
print(f"\n✅ Training Complete.")
print(f"Best Top-1 Accuracy: {best_acc:.2f}% at Epoch {convergence_epoch}")
print(f"Total Training Time: {total_time/60:.2f} minutes")

# =========================================================
# INFERENCE TIME
# =========================================================
model.eval()
dummy_input = torch.randn(1, 3, 32, 32).to(device)
start = time.time()
_ = model(dummy_input)
inference_time = time.time() - start
print(f"\n⚡ Inference Time (1 sample): {inference_time*1000:.3f} ms")

Epoch 1/100: 100%|██████████| 391/391 [00:11<00:00, 33.30it/s]



📊 Epoch 1:
Train Loss: 4.9397 | Train Acc@1: 1.81% | Train Acc@5: 7.82%
Val Loss: 4.6785 | Val Acc@1: 3.03% | Val Acc@5: 11.88%
Train–Test Gap: 1.22% | ECE: 0.0124 | MOC: 0.5049
⏱ Epoch Time: 13.97s


Epoch 2/100: 100%|██████████| 391/391 [00:10<00:00, 35.74it/s]



📊 Epoch 2:
Train Loss: 4.4789 | Train Acc@1: 2.84% | Train Acc@5: 11.94%
Val Loss: 4.3256 | Val Acc@1: 4.22% | Val Acc@5: 15.07%
Train–Test Gap: 1.38% | ECE: 0.0085 | MOC: 0.4110
⏱ Epoch Time: 13.22s


Epoch 3/100: 100%|██████████| 391/391 [00:11<00:00, 35.38it/s]


...


📊 Epoch 38:
Train Loss: 2.4061 | Train Acc@1: 36.83% | Train Acc@5: 69.23%
Val Loss: 2.3832 | Val Acc@1: 37.71% | Val Acc@5: 69.78%
Train–Test Gap: 0.88% | ECE: 0.0170 | MOC: 0.1873
⏱ Epoch Time: 14.11s


Epoch 39/100: 100%|██████████| 391/391 [00:11<00:00, 35.09it/s]



📊 Epoch 39:
Train Loss: 2.3863 | Train Acc@1: 37.20% | Train Acc@5: 69.63%
Val Loss: 2.5027 | Val Acc@1: 35.21% | Val Acc@5: 66.77%
Train–Test Gap: 1.99% | ECE: 0.0164 | MOC: 0.1932
⏱ Epoch Time: 13.41s


Epoch 40/100: 100%|██████████| 391/391 [00:11<00:00, 33.51it/s]



📊 Epoch 40:
Train Loss: 2.3760 | Train Acc@1: 37.32% | Train Acc@5: 69.84%
Val Loss: 2.4944 | Val Acc@1: 35.38% | Val Acc@5: 67.26%
Train–Test Gap: 1.94% | ECE: 0.0096 | MOC: 0.1880
⏱ Epoch Time: 14.56s

⏹ Early stopping triggered after 40 epochs (no improvement in 5 epochs).

✅ Training Complete.
Best Top-1 Accuracy: 40.19% at Epoch 35
Total Training Time: 9.46 minutes

⚡ Inference Time (1 sample): 56.292 ms
